In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time, os, joblib

import mlflow
import mlflow.sklearn
import mlflow.pytorch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, f1_score,
                             precision_score, recall_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"MLflow version: {mlflow.__version__}")

Device: cuda
MLflow version: 3.10.1


In [2]:
import os

# Build an absolute path to your project root — no more relative path confusion
PROJECT_ROOT = r"D:\study\projects\fraud-detection\fraud-detection"

# SQLite database file — stores all experiment data (fixes the FutureWarning)
DB_PATH      = os.path.join(PROJECT_ROOT, "mlflow.db")
TRACKING_URI = f"sqlite:///{DB_PATH}"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("fraud-detection")

print(f"Project root:  {PROJECT_ROOT}")
print(f"Tracking URI:  {TRACKING_URI}")
print(f"MLflow ready!")

Project root:  D:\study\projects\fraud-detection\fraud-detection
Tracking URI:  sqlite:///D:\study\projects\fraud-detection\fraud-detection\mlflow.db
MLflow ready!


In [3]:
X_train = pd.read_csv("../data/processed/X_train.csv").values
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
X_test  = pd.read_csv("../data/processed/X_test.csv").values
y_test  = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(f"Training: {X_train.shape} | Test: {X_test.shape}")

Training: (454902, 29) | Test: (56962, 29)


In [4]:
def log_metrics(y_true, y_pred, y_proba, prefix=""):
    """Calculate and return a dict of evaluation metrics."""
    return {
        f"{prefix}roc_auc":   roc_auc_score(y_true, y_proba),
        f"{prefix}f1":        f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall":    recall_score(y_true, y_pred),
    }

In [5]:
# Define the variants we want to compare
rf_variants = [
    {"n_estimators": 50,  "max_depth": 8,  "name": "rf-small"},
    {"n_estimators": 100, "max_depth": 10, "name": "rf-medium"},
    {"n_estimators": 200, "max_depth": 15, "name": "rf-large"},
]

for config in rf_variants:
    # mlflow.start_run() opens a new tracked experiment run
    with mlflow.start_run(run_name=config["name"]):
        
        # 1. Log parameters — the settings we chose
        mlflow.log_param("model_type",   "RandomForest")
        mlflow.log_param("n_estimators", config["n_estimators"])
        mlflow.log_param("max_depth",    config["max_depth"])
        mlflow.log_param("smote",        True)
        
        # 2. Train
        start = time.time()
        rf = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
        rf.fit(X_train, y_train)
        train_time = time.time() - start
        
        # 3. Predict
        preds = rf.predict(X_test)
        probs = rf.predict_proba(X_test)[:, 1]
        
        # 4. Log metrics — the results
        metrics = log_metrics(y_test, preds, probs)
        mlflow.log_metrics(metrics)
        mlflow.log_metric("train_time_seconds", train_time)
        
        # 5. Log the model itself — stored inside the MLflow run
        mlflow.sklearn.log_model(rf, "model")
        
        print(f"{config['name']:12s} | "
              f"ROC-AUC: {metrics['roc_auc']:.4f} | "
              f"F1: {metrics['f1']:.4f} | "
              f"Recall: {metrics['recall']:.4f} | "
              f"Time: {train_time:.1f}s")

2026/04/10 03:47:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:47:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-small     | ROC-AUC: 0.9784 | F1: 0.4198 | Recall: 0.8673 | Time: 5.5s


2026/04/10 03:47:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:47:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-medium    | ROC-AUC: 0.9777 | F1: 0.5714 | Recall: 0.8571 | Time: 11.4s


2026/04/10 03:47:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:47:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-large     | ROC-AUC: 0.9829 | F1: 0.7378 | Recall: 0.8469 | Time: 28.2s


In [6]:
class FraudDetectionNet(nn.Module):
    def __init__(self, input_size, hidden1=128, hidden2=64, hidden3=32, dropout1=0.3, dropout2=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(hidden2, hidden3),
            nn.BatchNorm1d(hidden3),
            nn.ReLU(),

            nn.Linear(hidden3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


def train_neural_net(config, X_train, y_train, X_test, y_test, device):
    """Train a neural network and return predictions + training history."""
    
    # Prepare tensors
    X_tr = torch.FloatTensor(X_train).to(device)
    y_tr = torch.FloatTensor(y_train).to(device)
    X_te = torch.FloatTensor(X_test).to(device)

    loader = DataLoader(TensorDataset(X_tr, y_tr),
                        batch_size=config["batch_size"], shuffle=True)

    model = FraudDetectionNet(
        input_size=X_train.shape[1],
        hidden1=config["hidden1"],
        hidden2=config["hidden2"],
        dropout1=config["dropout1"],
        dropout2=config["dropout2"]
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    loss_history = []

    for epoch in range(config["epochs"]):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in loader:
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(loader))

    # Evaluate
    model.eval()
    with torch.no_grad():
        probs = model(X_te).cpu().numpy()
    preds = (probs >= 0.5).astype(int)

    return model, preds, probs, loss_history

In [7]:
nn_variants = [
    {
        "name": "nn-small",
        "hidden1": 64,  "hidden2": 32,
        "dropout1": 0.3, "dropout2": 0.2,
        "lr": 0.001, "epochs": 15, "batch_size": 2048
    },
    {
        "name": "nn-medium",
        "hidden1": 128, "hidden2": 64,
        "dropout1": 0.3, "dropout2": 0.2,
        "lr": 0.001, "epochs": 20, "batch_size": 2048
    },
    {
        "name": "nn-large",
        "hidden1": 256, "hidden2": 128,
        "dropout1": 0.4, "dropout2": 0.3,
        "lr": 0.0005, "epochs": 25, "batch_size": 1024
    },
]

for config in nn_variants:
    with mlflow.start_run(run_name=config["name"]):

        # Log all config values as parameters
        for k, v in config.items():
            if k != "name":
                mlflow.log_param(k, v)
        mlflow.log_param("model_type", "NeuralNetwork")
        mlflow.log_param("device", str(device))

        # Train
        start = time.time()
        model, preds, probs, loss_history = train_neural_net(
            config, X_train, y_train, X_test, y_test, device
        )
        train_time = time.time() - start

        # Log metrics
        metrics = log_metrics(y_test, preds, probs)
        mlflow.log_metrics(metrics)
        mlflow.log_metric("train_time_seconds", train_time)

        # Log loss at each epoch so we can see the training curve in MLflow
        for epoch, loss_val in enumerate(loss_history):
            mlflow.log_metric("train_loss", loss_val, step=epoch)

        # Log the PyTorch model
        mlflow.pytorch.log_model(model, "model")

        print(f"{config['name']:12s} | "
              f"ROC-AUC: {metrics['roc_auc']:.4f} | "
              f"F1: {metrics['f1']:.4f} | "
              f"Recall: {metrics['recall']:.4f} | "
              f"Time: {train_time:.1f}s")

2026/04/10 03:49:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:49:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/04/10 03:49:06 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/10 03:49:09 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128).

nn-small     | ROC-AUC: 0.9745 | F1: 0.6227 | Recall: 0.8673 | Time: 42.4s


2026/04/10 03:50:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:50:01 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/04/10 03:50:01 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/10 03:50:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128).

nn-medium    | ROC-AUC: 0.9722 | F1: 0.6798 | Recall: 0.8776 | Time: 51.5s


2026/04/10 03:51:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 03:51:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/04/10 03:51:05 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/10 03:51:08 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128).

nn-large     | ROC-AUC: 0.9721 | F1: 0.7054 | Recall: 0.8673 | Time: 61.1s


In [8]:
from mlflow.tracking import MlflowClient

# Must use the exact same URI as set above
client = MlflowClient(tracking_uri=TRACKING_URI)

experiment = client.get_experiment_by_name("fraud-detection")

if experiment is None:
    print("Experiment not found — check that the tracking URI matches and runs completed.")
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.roc_auc DESC"]
    )

    if len(runs) == 0:
        print("No runs found — make sure the RF and NN training cells ran successfully.")
    else:
        print("All runs ranked by ROC-AUC:\n")
        print(f"{'Run name':<14} {'Model type':<15} {'ROC-AUC':>8} {'F1':>8} {'Recall':>8}")
        print("-" * 60)
        for run in runs:
            p = run.data.params
            m = run.data.metrics
            print(f"{run.info.run_name:<14} "
                  f"{p.get('model_type',''):<15} "
                  f"{m.get('roc_auc', 0):>8.4f} "
                  f"{m.get('f1', 0):>8.4f} "
                  f"{m.get('recall', 0):>8.4f}")

        best_run    = runs[0]
        best_run_id = best_run.info.run_id
        print(f"\nBest run: {best_run.info.run_name}  "
              f"(ROC-AUC: {best_run.data.metrics['roc_auc']:.4f})")

        model_uri  = f"runs:/{best_run_id}/model"
        registered = mlflow.register_model(model_uri, "fraud-detection-champion")
        print(f"Registered as: fraud-detection-champion (version {registered.version})")

Registered model 'fraud-detection-champion' already exists. Creating a new version of this model...
2026/04/10 03:51:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 585ac8d6e4a844ca984f01fdf466a1d5 has no artifacts at artifact path 'model', registering model based on models:/m-dfefed81137243ca97c44c37cad638f8 instead


All runs ranked by ROC-AUC:

Run name       Model type       ROC-AUC       F1   Recall
------------------------------------------------------------
rf-large       RandomForest      0.9829   0.7378   0.8469
rf-large       RandomForest      0.9829   0.7378   0.8469
rf-large       RandomForest      0.9829   0.7378   0.8469
rf-large       RandomForest      0.9829   0.7378   0.8469
nn-small       NeuralNetwork     0.9788   0.5695   0.8571
nn-large       NeuralNetwork     0.9785   0.6718   0.8878
rf-small       RandomForest      0.9784   0.4198   0.8673
rf-small       RandomForest      0.9784   0.4198   0.8673
rf-small       RandomForest      0.9784   0.4198   0.8673
rf-small       RandomForest      0.9784   0.4198   0.8673
rf-medium      RandomForest      0.9777   0.5714   0.8571
rf-medium      RandomForest      0.9777   0.5714   0.8571
rf-medium      RandomForest      0.9777   0.5714   0.8571
rf-medium      RandomForest      0.9777   0.5714   0.8571
nn-medium      NeuralNetwork     0.9772 

Created version '2' of model 'fraud-detection-champion'.
